In [0]:
silver_workforce = (spark.table("silver_workforce_fte"))
dim_time_gold = spark.table("date_dimension")
dim_region_gold = spark.table("region_dimension")
dim_ics_gold = spark.table("ics_dimension")
dim_organisation_gold = spark.table('org_dimension')
dim_staff_group_gold = spark.table("staff_dimension")

silver_workforce.show(10)

In [0]:
from pyspark.sql.functions import col

fact_workforce_gold = (
    silver_workforce.alias("w")

    # Date
    .join(
        dim_time_gold.alias("t"),
        col("w.month") == col("t.date"),
        how="left"
    )

    # Region
    .join(
        dim_region_gold.alias("r"),
        col("w.region_name") == col("r.region_name"),
        how="left"
    )

    # ICS
    .join(
        dim_ics_gold.alias("i"),
        col("w.ics_code") == col("i.ics_code"),
        how="left"
    )

    # Organisation
    .join(
        dim_organisation_gold.alias("o"),
        col("w.org_name") == col("o.org_name"),
        how="left"
    )

    # Staff group
    .join(
        dim_staff_group_gold.alias("s"),
        col("w.staff_group") == col("s.staff_group"),
        how="left"
    )

    .select(
        col("t.date_key"),
        col("r.region_key"),
        col("i.ics_key"),
        col("o.org_key"),
        col("s.staff_group_key"),
        col("w.fte")
    )
)

fact_workforce_gold = fact_workforce_gold.filter(
    col("ics_key").isNotNull() &
    col("org_key").isNotNull()
)

fact_workforce_gold.show(10)

In [0]:
(fact_workforce_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/fact_workforce/"
    ) \
    .saveAsTable(
        "fact_workforce"
    ))
